# Finish q3b Reconciliation — single-cell-runnable end-to-end notebook

End-to-end notebook to finish the dismantle Qwen reconciliation pipeline:
re-train the q3b Eagle6 long head (the only missing artifact), generate
its tau + frontier eval, produce the reconciliation summary that combines
q3b + q1p5, and export everything for download.

**Design principles** — learned from the run that lost the q3b head:

1. **No silent advance.** Every stage ends by hard-asserting its artifact
   exists, is loadable, and has expected shape. Failure halts the cell.
2. **Local backup.** After q3b long completes, the head is immediately
   pushed via `files.download()` to your browser machine. Survives any
   subsequent Drive disaster.
3. **Atomic writes + sha256 progress journal.** Every stage records its
   completion to `reconciliation_progress.json` so re-running is safe.
4. **Heartbeat output every 30s** for any long subprocess so silence
   is never ambiguous — you always see elapsed time + GPU util/mem,
   even when the subprocess hasn't printed for a while.
5. **No inline cleanup.** Do NOT paste `rm -rf` cells into this notebook.
   If Drive fills up, stop and triage.

Prereqs (all currently restored): q3b corpus (395 shards, 10 GB) and
`qwen3b_frozen.npz` on Drive; q1p5 long head + eval already on Drive at
`qwen_reconciliation/checkpoints/q1p5_e6_b2_wide_b2_h16_ff60_lr5e-4_rd020_cw20_long/`.

Wall-clock budget on an A100 40GB: ~3-4 hr for the retrain, plus ~20 min
for tau + frontier + export. T4 will be ~3x slower; A100/L4 strongly
preferred.


In [ ]:
# Cell 1 — Setup, deps, pre-flight checks.
# Halts loudly if any input is missing. Do NOT proceed past a FAIL.
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess, shutil, time, threading, signal
from pathlib import Path


def run(cmd, **kwargs):
    # Run a subprocess with inherited stdio. Use for one-shot fast commands.
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    env = os.environ.copy()
    env.update(kwargs.pop('env', {}) or {})
    env['PYTHONUNBUFFERED'] = '1'
    print('$ ' + ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, **kwargs)


def run_with_heartbeat(cmd, label='subproc', interval_sec=30):
    # Run a long subprocess with a heartbeat thread that prints elapsed
    # time + GPU util/mem every interval_sec seconds. This is so the
    # notebook cell shows visible progress even when the subprocess is
    # silent (e.g. during Drive-FUSE corpus pre-scan or model load).
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('$ ' + ' '.join(cmd), flush=True)

    proc = subprocess.Popen(cmd, env=env)
    print(f'[{label}] spawned pid={proc.pid} at {time.strftime("%H:%M:%S")}', flush=True)

    stop_evt = threading.Event()
    start = time.time()

    def heartbeat():
        while not stop_evt.is_set():
            if stop_evt.wait(interval_sec):
                return
            elapsed_m = (time.time() - start) / 60
            # GPU snapshot (5s timeout so a hung nvidia-smi doesn't wedge the heartbeat)
            try:
                out = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                     '--format=csv,noheader,nounits'],
                    text=True, timeout=5,
                ).strip()
                util, mu, mt = [x.strip() for x in out.split(',')]
                gpu = f'GPU util={util}%  mem={mu}/{mt} MB'
            except Exception as e:
                gpu = f'GPU query failed: {e}'
            still_alive = (proc.poll() is None)
            alive_str = 'RUNNING' if still_alive else f'EXITED rc={proc.returncode}'
            print(f'[{label}] {alive_str}  elapsed={elapsed_m:.1f}m  {gpu}', flush=True)

    hb = threading.Thread(target=heartbeat, daemon=True)
    hb.start()
    try:
        rc = proc.wait()
    finally:
        stop_evt.set()
        hb.join(timeout=2)
    elapsed_m = (time.time() - start) / 60
    print(f'[{label}] finished rc={rc} in {elapsed_m:.1f}m', flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc


def kill_stale_trainers():
    # Kill any leftover eagle5_train_pytorch.py processes from a previous
    # attempt. Restarting a cell while the prior subprocess is still alive
    # causes GPU OOM and confusing silent behaviour.
    try:
        out = subprocess.check_output(['pgrep', '-f', 'eagle5_train_pytorch.py'],
                                       text=True).strip()
        pids = [int(p) for p in out.splitlines() if p.strip()]
    except subprocess.CalledProcessError:
        return  # no matches
    if pids:
        print(f'[kill-stale] killing leftover trainers: {pids}', flush=True)
        for pid in pids:
            try:
                os.kill(pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        time.sleep(3)
        # SIGKILL stragglers
        for pid in pids:
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        print(f'[kill-stale] done', flush=True)


run([sys.executable, '-m', 'pip', 'install', '-q',
     'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17',
     'safetensors>=0.4', 'tqdm>=4.66', 'hf_transfer>=0.1.9'])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTHONUNBUFFERED', '1')

import numpy as np
import torch
import transformers

assert torch.cuda.is_available(), 'No CUDA: Runtime -> Change runtime type -> GPU'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB  transformers={transformers.__version__}')

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/dismantle'])
else:
    run(['git', '-C', '/content/dismantle', 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', '/content/dismantle', 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir('/content/dismantle')
HEAD_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'repo @ {HEAD_SHA}')

# ---- Paths -------------------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/dismantle')
ARTIFACT_DIR = DRIVE_ROOT / 'qwen_reconciliation'
CKPT_ROOT = ARTIFACT_DIR / 'checkpoints'
EXPORT_DIR = Path('/content/drive/MyDrive/dismantle_export')
LOCAL_FROZEN = Path('/content/q3b_frozen.npz')
PROGRESS_PATH = ARTIFACT_DIR / 'reconciliation_progress.json'

# q3b winner config — exact hyperparams from the directory name of the
# original grid winner. b1_wide, 16 heads, ff_mult=6.0, lr=5e-4, rd=0.020,
# cw=0.20, 20 epochs, max_rows=8000, max_row_tokens=192.
Q3B_LONG_DIR_NAME = 'q3b_e6_b1_wide_b1_h16_ff60_lr5e-4_rd020_cw20_long'
Q3B_LONG_CKPT_DIR = CKPT_ROOT / Q3B_LONG_DIR_NAME
Q3B_HEAD_PATH = Q3B_LONG_CKPT_DIR / 'head_final.safetensors'
Q3B_SPEC = {
    'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 6.0,
    'lr': 5e-4, 'residual_delta': 0.020, 'calib_weight': 0.20,
    'epochs': 20, 'max_rows': 8000, 'max_row_tokens': 192,
}

Q1P5_LONG_DIR_NAME = 'q1p5_e6_b2_wide_b2_h16_ff60_lr5e-4_rd020_cw20_long'
Q1P5_LONG_CKPT_DIR = CKPT_ROOT / Q1P5_LONG_DIR_NAME

CORPUS_DIR = DRIVE_ROOT / 'qwen3b_corpus'
FROZEN_PATH = ARTIFACT_DIR / 'qwen3b_frozen.npz'
CAPTURE_LAYER_Q3B = 32

# ---- Pre-flight --------------------------------------------------------
print('\n=== Pre-flight ===')

failures = []
def check(label, ok, detail=''):
    mark = 'OK  ' if ok else 'FAIL'
    print(f'  [{mark}] {label}  {detail}')
    if not ok:
        failures.append(label)
    return ok

shards = sorted(CORPUS_DIR.glob('shard_*.parquet'))
check('q3b corpus shards', len(shards) >= 350, f'count={len(shards)}')
check('q3b frozen', FROZEN_PATH.exists(),
      f'size={FROZEN_PATH.stat().st_size/1e9:.2f} GB' if FROZEN_PATH.exists() else 'missing')
check('AWQ per-tensor', (ARTIFACT_DIR / 'qwen3b_awq.json').exists())
check('AWQ per-channel', (ARTIFACT_DIR / 'qwen3b_awq_per_channel.json').exists())
check('Q2 importance', (ARTIFACT_DIR / 'qwen3b_q2_importance.npz').exists())

q1p5_head = Q1P5_LONG_CKPT_DIR / 'head_final.safetensors'
check('q1p5 long head', q1p5_head.exists(),
      f'size={q1p5_head.stat().st_size/1e9:.2f} GB' if q1p5_head.exists() else 'missing')

if failures:
    raise SystemExit(
        f'Pre-flight failed: {failures}. Fix the FAIL items above and restart this cell.'
    )

# Stage frozen locally for fast trainer access (avoids Drive FUSE thrash).
if not LOCAL_FROZEN.exists() or LOCAL_FROZEN.stat().st_size != FROZEN_PATH.stat().st_size:
    print(f'\nStaging frozen -> {LOCAL_FROZEN} ...')
    shutil.copyfile(FROZEN_PATH, LOCAL_FROZEN)
print(f'frozen local: {LOCAL_FROZEN} ({LOCAL_FROZEN.stat().st_size/1e9:.2f} GB)')

# Colab keep-alive (clicks reconnect button every 60s).
try:
    from IPython.display import Javascript, display
    display(Javascript('''
if (window.__dismantle_ka) clearInterval(window.__dismantle_ka);
window.__dismantle_ka = setInterval(()=>{
    const b = document.querySelector("colab-toolbar-button#connect")
          || document.querySelector("colab-connect-button");
    if (b) b.click();
}, 60000);
'''))
    print('Installed Colab keep-alive')
except Exception as e:
    print(f'(keep-alive skipped: {e})')

print('\nPre-flight passed.')


In [ ]:
# Cell 2 — Progress journal helpers + resume state.
# Each stage records its output path, size, sha256 in reconciliation_progress.json.
# On re-run, stage_complete() lets us skip work that already finished.

def sha256_file(path, bs=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(bs)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_progress():
    if PROGRESS_PATH.exists():
        try:
            return json.loads(PROGRESS_PATH.read_text())
        except Exception:
            return {}
    return {}


def save_progress(prog):
    PROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = PROGRESS_PATH.with_suffix('.tmp')
    tmp.write_text(json.dumps(prog, indent=2, sort_keys=True))
    os.replace(tmp, PROGRESS_PATH)


def stage_complete(prog, stage):
    e = prog.get(stage)
    if not e:
        return False
    path_str = e.get('path') or e.get('summary_path') or e.get('manifest_path')
    if not path_str:
        return False
    p = Path(path_str)
    if not p.exists():
        return False
    if 'size' in e and p.stat().st_size != e['size']:
        return False
    return True


prog = load_progress()
print('Existing progress journal:')
if not prog:
    print('  (none — fresh run)')
for k, v in prog.items():
    bytes_ = v.get('size', 0)
    sz = f'{bytes_/1e9:.2f} GB' if bytes_ else ''
    print(f'  {k}: {v.get("path", v.get("summary_path", v.get("manifest_path", "?")))} {sz}')


In [ ]:
# Cell 3 — q3b long retrain.
# Skip if head_final.safetensors exists and matches the journaled sha256.
# Else: train with the exact winner hyperparams; on completion, verify the
# safetensors is loadable, sha256 + record progress, AND trigger an
# immediate local-machine download.
#
# A heartbeat thread prints elapsed time + GPU util/mem every 30s so you
# can tell the cell is alive even when the trainer is silent during
# Drive-corpus pre-scan (~2-5 min) or model load (~30s).

STAGE = 'q3b_long_head'

if stage_complete(prog, STAGE):
    print(f'[{STAGE}] already complete — skipping')
    print(f'  path = {prog[STAGE]["path"]}')
    print(f'  sha256 = {prog[STAGE].get("sha256", "?")[:16]}...')
else:
    # Kill any leftover trainer process from a prior cell run.
    kill_stale_trainers()
    # Free GPU memory in case the runtime is shared with a previous attempt.
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    free_mb = (torch.cuda.mem_get_info()[0]) / 1024**2 if torch.cuda.is_available() else 0
    print(f'GPU free before launch: {free_mb:.0f} MB')

    Q3B_LONG_CKPT_DIR.mkdir(parents=True, exist_ok=True)

    # VRAM-tiered batch size.
    if VRAM_GB >= 70:
        batch = 24
    elif VRAM_GB >= 35:
        batch = 16
    elif VRAM_GB >= 20:
        batch = 8
    else:
        batch = 4
    print(f'Training q3b long: batch={batch}, spec={Q3B_SPEC}')
    print(f'Expected cold-start silence: ~3-7 min before first [data] line.')
    print(f'Heartbeat will print every 30s with GPU util + memory.')
    print(f'If heartbeat shows GPU util 0% and RUNNING for >5 min, the trainer is hung.')
    print()

    cmd = [
        sys.executable, 'colab/eagle5_train_pytorch.py',
        '--corpus-dir', str(CORPUS_DIR),
        '--frozen', str(LOCAL_FROZEN),
        '--ckpt-dir', str(Q3B_LONG_CKPT_DIR),
        '--epochs', str(Q3B_SPEC['epochs']),
        '--batch-size', str(batch),
        '--seq-len', '16',
        '--lr', str(Q3B_SPEC['lr']),
        '--capture-layer', str(CAPTURE_LAYER_Q3B),
        '--max-rows', str(Q3B_SPEC['max_rows']),
        '--max-row-tokens', str(Q3B_SPEC['max_row_tokens']),
        '--sparsity-head', 'off',
        '--seed', '0',
        '--calib-loss-weight', str(Q3B_SPEC['calib_weight']),
        '--residual-delta-loss-weight', str(Q3B_SPEC['residual_delta']),
        '--num-blocks', str(Q3B_SPEC['num_blocks']),
        '--head-heads', str(Q3B_SPEC['head_heads']),
        '--head-ff-mult', str(Q3B_SPEC['head_ff_mult']),
        '--save-safetensors',
    ]
    start = time.time()
    run_with_heartbeat(cmd, label='train', interval_sec=30)
    elapsed_min = (time.time() - start) / 60

    # Hard-assert head exists, loadable, expected size range.
    assert Q3B_HEAD_PATH.exists(), f'CRITICAL: trainer did not write {Q3B_HEAD_PATH}'
    sz = Q3B_HEAD_PATH.stat().st_size
    assert 0.5e9 < sz < 4e9, f'unexpected head size: {sz/1e9:.2f} GB (out of 0.5–4 GB band)'

    from safetensors.torch import load_file as _load
    sd = _load(str(Q3B_HEAD_PATH))
    n_params = sum(v.numel() for v in sd.values())
    n_tensors = len(sd)
    assert n_params > 1_000_000, f'head has too few params: {n_params}'
    assert n_tensors >= 4, f'head missing tensors: {n_tensors}'
    print(f'  verified: {n_params:,} params across {n_tensors} tensors')

    sha = sha256_file(Q3B_HEAD_PATH)
    prog[STAGE] = {
        'path': str(Q3B_HEAD_PATH),
        'size': sz,
        'sha256': sha,
        'n_params': int(n_params),
        'n_tensors': int(n_tensors),
        'train_minutes': round(elapsed_min, 1),
        'spec': Q3B_SPEC,
        'repo_sha': HEAD_SHA,
        'completed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    save_progress(prog)
    print(f'\n[{STAGE}] complete  sha256={sha[:16]}...  {sz/1e9:.2f} GB  in {elapsed_min:.1f} min')

    # Immediate local-machine backup — survives any Drive disaster.
    print('\nTriggering local download of the head — accept the browser prompt.')
    print('(If your browser blocks it, download manually from the Files pane.)')
    try:
        from google.colab import files as _files
        _files.download(str(Q3B_HEAD_PATH))
    except Exception as e:
        print(f'  auto-download failed: {e}')
        print(f'  Manual fallback path: {Q3B_HEAD_PATH}')


In [ ]:
# Cell 4 — q3b tau eval. Heartbeat every 30s.
STAGE = 'q3b_tau'
TAU_PATH = Q3B_LONG_CKPT_DIR / 'tau.json'

if stage_complete(prog, STAGE):
    print(f'[{STAGE}] already complete')
else:
    run_with_heartbeat([
        sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
        '--ckpt', str(Q3B_HEAD_PATH),
        '--frozen', str(LOCAL_FROZEN),
        '--corpus', str(CORPUS_DIR),
        '--out', str(TAU_PATH),
        '--depth', '4',
        '--max-windows', '6000',
        '--max-row-tokens', str(Q3B_SPEC['max_row_tokens']),
        '--base-tps', '27.0',
        '--w4a8-multiplier', '1.15',
        '--spec-efficiency', '0.70',
    ], label='tau')
    assert TAU_PATH.exists() and TAU_PATH.stat().st_size > 100, f'tau write failed: {TAU_PATH}'
    tau = json.loads(TAU_PATH.read_text())
    proj_tps = tau.get('projection', {}).get('projected_dec_tps')
    prog[STAGE] = {
        'path': str(TAU_PATH),
        'size': TAU_PATH.stat().st_size,
        'sha256': sha256_file(TAU_PATH),
        'tau': tau.get('tau'),
        'depth1_accept_rate': tau.get('depth1_accept_rate'),
        'projected_dec_tps': proj_tps,
        'completed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    save_progress(prog)
    print(f'[{STAGE}] tau={tau.get("tau"):.3f}  '
          f'depth1_accept={tau.get("depth1_accept_rate"):.3f}  '
          f'projected_dec_tps={proj_tps:.1f}')


In [ ]:
# Cell 5 — q3b frontier policy search. Heartbeat every 30s.
STAGE = 'q3b_frontier'
FRONTIER_PATH = Q3B_LONG_CKPT_DIR / 'frontier.json'

if stage_complete(prog, STAGE):
    print(f'[{STAGE}] already complete')
else:
    run_with_heartbeat([
        sys.executable, 'colab/eagle5_frontier_policy.py',
        '--ckpt', str(Q3B_HEAD_PATH),
        '--frozen', str(LOCAL_FROZEN),
        '--corpus', str(CORPUS_DIR),
        '--out', str(FRONTIER_PATH),
        '--max-depth', '12',
        '--depths', '4,6,8,12',
        '--lattice-widths', '2,3,4',
        '--max-windows', '6000',
        '--max-row-tokens', str(Q3B_SPEC['max_row_tokens']),
        '--base-tps', '27.0',
        '--w4a8-multiplier', '1.15',
        '--spec-efficiency', '0.70',
    ], label='frontier')
    assert FRONTIER_PATH.exists() and FRONTIER_PATH.stat().st_size > 100
    fr = json.loads(FRONTIER_PATH.read_text())
    best = fr.get('policies', {}).get('best_deployable', {})
    prog[STAGE] = {
        'path': str(FRONTIER_PATH),
        'size': FRONTIER_PATH.stat().st_size,
        'sha256': sha256_file(FRONTIER_PATH),
        'best_deployable': best,
        'completed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    save_progress(prog)
    print(f'[{STAGE}] kind={best.get("kind")} '
          f'projected_dec_tps={best.get("projected_dec_tps"):.1f} '
          f'accept/verify={best.get("accepted_draft_tokens_per_verify", 0):.2f}')


In [ ]:
# Cell 6 — Reconciliation summary (q3b + q1p5).
STAGE = 'reconciliation_summary'
SUMMARY_PATH = ARTIFACT_DIR / 'reconciliation_summary.md'
WINNERS_PATH = ARTIFACT_DIR / 'reconciliation_frontier_winners.json'

q3b_fr = json.loads((Q3B_LONG_CKPT_DIR / 'frontier.json').read_text())
q3b_tau = json.loads((Q3B_LONG_CKPT_DIR / 'tau.json').read_text())

q1p5_fr_path = Q1P5_LONG_CKPT_DIR / 'frontier.json'
q1p5_tau_path = Q1P5_LONG_CKPT_DIR / 'tau.json'
q1p5_fr = json.loads(q1p5_fr_path.read_text()) if q1p5_fr_path.exists() else None
q1p5_tau = json.loads(q1p5_tau_path.read_text()) if q1p5_tau_path.exists() else None

winners = {'frontier_per_target': {
    'q3b': {
        'tag': Q3B_LONG_DIR_NAME,
        'head_path': str(Q3B_HEAD_PATH),
        'best_deployable': q3b_fr.get('policies', {}).get('best_deployable', {}),
        'runtime_hints': q3b_fr.get('runtime_hints', {}),
        'tau': q3b_tau.get('tau'),
    }
}}
if q1p5_fr is not None:
    winners['frontier_per_target']['q1p5'] = {
        'tag': Q1P5_LONG_DIR_NAME,
        'head_path': str(Q1P5_LONG_CKPT_DIR / 'head_final.safetensors'),
        'best_deployable': q1p5_fr.get('policies', {}).get('best_deployable', {}),
        'runtime_hints': q1p5_fr.get('runtime_hints', {}),
        'tau': q1p5_tau.get('tau') if q1p5_tau else None,
    }
tmp = WINNERS_PATH.with_suffix('.tmp')
tmp.write_text(json.dumps(winners, indent=2, sort_keys=True))
os.replace(tmp, WINNERS_PATH)

lines = [
    '# Qwen Reconciliation Summary',
    '',
    f'Generated `{time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}` on `{GPU_NAME}`.',
    f'repo @ `{HEAD_SHA}` via `colab/finish_q3b_reconciliation.ipynb`.',
    '',
    '## Long-trained Eagle6 heads',
    '',
]
for tt, info in winners['frontier_per_target'].items():
    bd = info['best_deployable']
    tau_str = f' · tau=`{info["tau"]:.3f}`' if info.get('tau') is not None else ''
    lines.append(
        f'- **{tt}**: `{info["tag"]}` · '
        f'projected `{bd.get("projected_dec_tps", 0):.1f}` dec_tps · '
        f'kind=`{bd.get("kind", "?")}` · '
        f'accept/verify=`{bd.get("accepted_draft_tokens_per_verify", 0):.2f}`'
        f'{tau_str}'
    )
    lines.append(f'  - head: `{info["head_path"]}`')

lines.extend([
    '',
    '## Notes',
    '',
    'These heads do not deliver dec_tps speedup on Mac yet — spec-decode is',
    'wired into `crates/dismantle-core/src/model/deepseek_v2.rs` only;',
    '`qwen_dense.rs` has zero Eagle5 references. The trained heads are',
    'inventory waiting on the Rust port (see `docs/eagle5_qwen_port_plan.md`).',
    '',
])
tmp = SUMMARY_PATH.with_suffix('.tmp')
tmp.write_text('\n'.join(lines) + '\n')
os.replace(tmp, SUMMARY_PATH)

prog[STAGE] = {
    'summary_path': str(SUMMARY_PATH),
    'size': SUMMARY_PATH.stat().st_size,
    'winners_path': str(WINNERS_PATH),
    'completed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
save_progress(prog)

print(SUMMARY_PATH.read_text())


In [ ]:
# Cell 7 — Export essentials, then download zip locally.
STAGE = 'export'

run([
    sys.executable, 'colab/export_reconciliation_essentials.py',
    '--drive-root', str(DRIVE_ROOT),
    '--export-dir', str(EXPORT_DIR),
    '--strict',
    '--zip',
])

manifest_path = EXPORT_DIR / 'manifest.json'
assert manifest_path.exists(), f'export manifest missing: {manifest_path}'
manifest = json.loads(manifest_path.read_text())

required = ('q3b_head', 'q1p5_head')
missing_req = [k for k in required if k in manifest.get('missing', {})]
assert not missing_req, f'CRITICAL: export missing required heads: {missing_req}'

print('\n=== Export manifest ===')
print(f'  files: {len(manifest["files"])}  missing optional: {len(manifest.get("missing", {}))}')
for k, v in sorted(manifest['files'].items()):
    print(f'   OK  {k}  {v["bytes"]/1e9:.3f} GB -> {v["exported"]}')
for k in sorted(manifest.get('missing', {})):
    print(f'   ..  {k}  (optional, not present)')

zip_path = EXPORT_DIR.with_suffix('.zip')
if zip_path.exists():
    print(f'\nLocal-download zip: {zip_path} ({zip_path.stat().st_size/1e9:.2f} GB)')
    try:
        from google.colab import files as _files
        _files.download(str(zip_path))
    except Exception as e:
        print(f'  auto-download failed: {e}')
        print(f'  Manual fallback: download {zip_path} from the Files pane.')
else:
    print(f'  (zip not produced; --zip may have failed silently)')

prog[STAGE] = {
    'manifest_path': str(manifest_path),
    'size': manifest_path.stat().st_size,
    'zip_path': str(zip_path) if zip_path.exists() else None,
    'completed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
save_progress(prog)

print('\n=== DONE ===')
print(f'Drive export: {EXPORT_DIR}')
if zip_path.exists():
    print(f'Local zip:    {zip_path}')
print(f'Progress journal: {PROGRESS_PATH}')
